# MeshVTON v2\n\nHücreler: kurulum / veri / Faz 1 zero-shot / sentetik duman testi / sonuçlar.\n**Mantık notebook'ta değil `v2/` paketindedir** — değişiklik script/modülde yapılır, hücrede değil.\n\n⚠️ Bu notebook'u Drive'daki kopyadan değil, `git pull` sonrası repo'dan (`/content/MeshVTON/v2/notebooks/`) açın.

In [ ]:
#@title 1) Kurulum (~2-3 dk; derleme YOK, pytorch3d GEREKMEZ)
import os
if not os.path.exists('/content/MeshVTON'):
    !git clone https://github.com/SerhanTelatar/MeshVTON /content/MeshVTON
%cd /content/MeshVTON
!git pull

# Yalnız eksik paketler (torch/transformers/accelerate Colab'da hazır):
!pip -q install "diffusers>=0.34" "peft>=0.14" lpips einops sentencepiece trimesh smplx pyrender onnxruntime
os.environ['PYOPENGL_PLATFORM'] = 'egl'   # pyrender headless GPU

# IDM-VTON yalnız KIŞI ön-işlemesi için (parsing onnx + openpose ckpt); densepose/detectron2 YOK
if not os.path.exists('/content/IDM-VTON'):
    !git clone -q https://github.com/yisol/IDM-VTON /content/IDM-VTON
# Ön-işleme model dosyaları: wget DEĞİL hf_hub_download (LFS'i doğru indirir; wget
# HF'ten HTML hata sayfası kaydedip INVALID_PROTOBUF'a yol açabiliyor)
import shutil
from huggingface_hub import hf_hub_download
for repo_path in ('humanparsing/parsing_atr.onnx',
                  'humanparsing/parsing_lip.onnx',
                  'openpose/ckpts/body_pose_model.pth'):
    local = f'/content/IDM-VTON/ckpt/{repo_path}'
    if not (os.path.exists(local) and os.path.getsize(local) > 1_000_000):
        os.makedirs(os.path.dirname(local), exist_ok=True)
        shutil.copy(hf_hub_download('yisol/IDM-VTON', repo_path), local)
    assert os.path.getsize(local) > 1_000_000, f'bozuk indirme: {local}'

# FLUX.1 dev gated: Colab Secrets'a HF_TOKEN ekleyin (soldaki anahtar simgesi, Notebook access açık)
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('Kurulum OK')

In [ ]:
#@title 2) Veri — Drive/MeshVTON düzenine göre (images.zip, garments_3d.zip, smplx/, smplx_params.zip)
import os
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/MeshVTON'

# zip'lerin İÇİNDE üst klasör var (images/, garments_3d/) -> bir üst hedefe aç:
!mkdir -p /content/MeshVTON/data/raw
!unzip -q -n $D/images.zip      -d /content/MeshVTON/data/raw    # -> data/raw/images/
!unzip -q -n $D/garments_3d.zip -d /content/MeshVTON/data        # -> data/garments_3d/

# SMPL-X modeli (Drive: MeshVTON/smplx/SMPLX_NEUTRAL.*) — 'checkpoints' klasörünü BİZ oluşturuyoruz,
# Drive'da olması gerekmiyor:
!mkdir -p /content/MeshVTON/checkpoints/pretrained/smplx
!cp $D/smplx/SMPLX_NEUTRAL.* /content/MeshVTON/checkpoints/pretrained/smplx/
os.environ['SMPLX_MODEL_DIR'] = '/content/MeshVTON/checkpoints/pretrained/smplx'

# Poz bankası: v1'in kişi başına SMPL-X tahminleri (sentetik veri gerçek pozlarla üretilsin)
!unzip -q -n $D/smplx_params.zip -d /content/MeshVTON/data/smplx_params

# doğrulama
!ls /content/MeshVTON/data/raw/images | head -3
!find /content/MeshVTON/data/garments_3d -name "*.obj" | wc -l
!ls /content/MeshVTON/checkpoints/pretrained/smplx
!ls /content/MeshVTON/data/smplx_params | head -3

In [ ]:
#@title 3) Faz 1 — Zero-shot taban çizgisi (golden set + FLUX)
VITONHD_TEST = "/content/MeshVTON/data/raw/images"       #@param {type:"string"}
GARMENTS     = "/content/MeshVTON/data/garments_3d"      #@param {type:"string"}
VARIANT      = "fill_spatial"  #@param ["fill_spatial", "kontext"]
LIMIT        = 4  #@param {type:"integer"}  # 0 = tam koşu (100 kombo)

import os
if not os.path.exists('v2/data/golden/manifest.json'):
    !python v2/scripts/build_golden_set.py --vitonhd-test "$VITONHD_TEST" --garments "$GARMENTS"

limit_arg = f"--limit {LIMIT}" if LIMIT else ""
!python v2/scripts/zero_shot_baseline.py --variant $VARIANT --idm-repo /content/IDM-VTON $limit_arg

In [ ]:
#@title 4) Faz 2/3 — Sentetik veri duman testi (5 örnek × 4 görüş)
NUM = 5  #@param {type:"integer"}
!python v2/scripts/generate_synthetic.py \
    --garments /content/MeshVTON/data/garments_3d \
    --poses    /content/MeshVTON/data/smplx_params \
    --num $NUM --limit-garments 3

In [ ]:
#@title 5) Sonuçlar
from IPython.display import Image as I, display, Markdown
import pathlib

# Faz 1 raporları
for v in ("fill_spatial", "kontext"):
    md = pathlib.Path(f"v2/eval_results/phase1_{v}.md")
    grid = pathlib.Path(f"v2/eval_results/phase1_{v}_grid.png")
    if md.exists():
        display(Markdown(f"## Faz 1: {v}"), Markdown(md.read_text()))
    if grid.exists():
        display(I(str(grid), width=900))

# Sentetik duman testi: ilk örneğin 4 görüşü (QA: drape oturuyor mu, 180° gerçekten arka mı?)
synth = sorted(pathlib.Path("v2/data/synth").glob("s*/"))
if synth:
    s = synth[0]
    display(Markdown(f"## Sentetik QA: {s.name}"), I(str(s / "appearance_ref.png"), width=200))
    for v in ("000", "090", "180", "270"):
        p = s / f"view_{v}" / "gt.png"
        if p.exists():
            display(Markdown(f"**{int(v)}°** (gt / normal / depth+sil / agnostic)"))
            for f in ("gt", "normal", "depth_sil", "agnostic"):
                display(I(str(s / f"view_{v}" / f"{f}.png"), width=180))